In [26]:
from odc.stac import stac_load
from pystac_client import Client
from IPython.display import display

catalog = Client.open("https://earth-search.aws.element84.com/v1")


km2deg = 1.0 / 111
x, y = (113.887, -25.843)  # Center point of a query
r = 100 * km2deg
bbox = (x - r, y - r, x + r, y + r)

crs = "epsg:3857"
zoom = 2**5  # overview level 5

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2023-01-01/2023-01-31",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=100,
)

items = list(search.get_items())

# Load all 13 Sentinel-2 bands
all_bands = [
    "coastal", "blue", "green", "red", "rededge1", "rededge2", "rededge3",
    "nir", "nir08", "nir09", "swir16", "swir22"
]

ds = stac_load(
    items,
    bands=all_bands,
    crs=crs,
    resolution=10 * zoom,
    chunks={},  # <-- use Dask
    groupby="solar_day",
)
display(ds)

/home/cy0/.local/lib/python3.10/site-packages/pystac_client/item_search.py:888: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


<xarray.Dataset> Size: 346MB
Dimensions:      (y: 1101, x: 1092, time: 12)
Coordinates:
  * y            (y) float64 9kB -2.797e+06 -2.798e+06 ... -3.149e+06 -3.149e+06
  * x            (x) float64 9kB 1.247e+07 1.247e+07 ... 1.282e+07 1.282e+07
    spatial_ref  int32 4B 3857
  * time         (time) datetime64[ns] 96B 2023-01-02T02:45:09.869000 ... 202...
Data variables:
    coastal      (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    blue         (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    green        (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    red          (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    rededge1     (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    rededge2     (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    rededge3     (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    nir          (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    nir08        (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    nir09        (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    swir16       (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>
    swir22       (time, y, x) uint16 29MB dask.array<chunksize=(1, 1101, 1092), meta=np.ndarray>

In [27]:
!wget -nc https://huggingface.co/datasets/neurograce/SubstationDataset/resolve/main/models/swin_single_image_multi_spectral.pth -P ./weights/

File ‘./weights/swin_single_image_multi_spectral.pth’ already there; not retrieving.



In [28]:
import utils
import random
args = utils.parse_arguments(False)

args.batch_size = 1
args.worker = 1
args.in_channels = 9
args.upsampled_image_size=224
args.upsampled_mask_size=224
args.normalizing_type= 'zscore'
args.model_type='swin'
args.vit_size='base'
args.use_timepoints=False
args.resume_training=False
args.checkpoint = './weights/swin_single_image_multi_spectral.pth'
args = utils.sanity_checks(args)

{'data_dir': '/scratch/kj1447/gracelab/dataset', 'dataset': 'substation', 'model_dir': '/scratch/kj1447/gracelab/models/SWIN_FPN_low_LR', 'loss': 'BCE', 'alpha': 0.25, 'epochs': 250, 'batch_size': 16, 'workers': 16, 'train_ratio': 0.8, 'learning_rate': 0.0001, 'seed': 42, 'resume_training': False, 'checkpoint': None, 'lookback': 10, 'starting_epoch': 0, 'upsampled_image_size': 256, 'upsampled_mask_size': 256, 'model_type': 'swin', 'in_channels': 3, 'use_timepoints': True, 'pretrained': True, 'normalizing_type': 'constant', 'normalizing_factor': 4000, 'learned_upsampling': True, 'exp_name': 'SWIN_MI', 'exp_number': 1, 'vit_size': 'base', 'task': 'building', 'type_of_model': 'classification', 'timepoint_aggregation': 'concat', 'pretrained_weights': 'Sentinel2_SwinB_MI_RGB', 'mask_2d': True}


In [29]:
#DATALOADER
import torchvision.transforms as transforms
geo_transform = None
color_transform = None

image_resize = transforms.Compose([transforms.Resize(args.upsampled_image_size,transforms.InterpolationMode.BICUBIC, antialias=True)])

In [31]:
import torch
from models import setup_model

if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

# Load the pretrained weights
checkpoint = torch.load("weights/swin_single_image_multi_spectral.pth", map_location="cpu")

torch.manual_seed(args.seed)
model = setup_model(args)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [25]:
model.load_state_dict(checkpoint['model_state_dict'])

# Set the model to evaluation mode
model.eval()








RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.